# Explore the Feature-GBM baseline

Run `uv run legaleval-gbm` first: this notebook reads its files from `outputs/` and the raw texts from `../data/`.

The goal is to *look* at the data and the model's mistakes, which is where ideas for the next features come from.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from legaleval_gbm.data import BLOCK_LABELS, DEFAULT_DATA_DIR, TARGET, load_split
from legaleval_gbm.metric import competition_score

OUT = Path("../outputs")
pd.set_option("display.max_colwidth", 200)

train = load_split(DEFAULT_DATA_DIR, "train")
features = pd.read_csv(OUT / "features_train.csv")
oof = pd.read_csv(OUT / "oof_gbm.csv")
importance = pd.read_csv(OUT / "importance_gbm.csv")

df = train.merge(features.drop(columns=["id_sud_resh", "block_name", TARGET]), on="row_id").merge(
    oof.drop(columns=[TARGET]), on="row_id"
)
df["block"] = df["block_name"].map(BLOCK_LABELS)
df["error"] = df["oof_rescaled"] - df[TARGET]
df.shape

## 1. How experts score each block

The block alone is the strongest feature. Wide boxes are blocks where the extraction quality really varies, which is where text features have to do the work.

In [ ]:
order = df.groupby("block")[TARGET].mean().sort_values().index
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([df.loc[df.block == b, TARGET] for b in order], tick_labels=order, orientation="horizontal")
ax.set_xlabel("expert_score_mean (1-5)")
ax.set_title("Expert score by block")
plt.tight_layout()

## 2. Out-of-fold predictions against the truth

Each point is a training row predicted by a model that never saw its case. A perfect model puts every point on the diagonal.

In [ ]:
print("raw     ", competition_score(df[TARGET], df["oof_raw"]))
print("rescaled", competition_score(df[TARGET], df["oof_rescaled"]))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(df[TARGET], df["oof_rescaled"], alpha=0.35, s=14)
ax.plot([1, 5], [1, 5], color="grey", linewidth=1)
ax.set_xlabel("expert score")
ax.set_ylabel("predicted (OOF, rescaled)")
plt.tight_layout()

Scores averaged over seeds are a little optimistic compared with the per-seed numbers the CLI prints: averaging smooths the noise. Treat the CLI table as the honest number.

## 3. Which features the model relies on

Permutation importance: shuffle one feature on held-out folds and measure how much the error grows. Near-zero or negative means the model does not need it.

In [ ]:
importance

In [ ]:
feature = "trigram_copy_ratio"
fig, axes = plt.subplots(1, 7, figsize=(18, 3), sharey=True)
for ax, b in zip(axes, order, strict=True):
    part = df[df.block == b]
    ax.scatter(part[feature], part[TARGET], alpha=0.4, s=10)
    ax.set_title(b, fontsize=9)
    ax.set_xlabel(feature, fontsize=8)
axes[0].set_ylabel("expert score")
plt.tight_layout()

## 4. Where the model is wrong, per block

In [ ]:
df.groupby("block").agg(
    rows=(TARGET, "size"),
    expert_mean=(TARGET, "mean"),
    mae=("error", lambda e: e.abs().mean()),
    corr=("oof_rescaled", lambda p: p.corr(df.loc[p.index, TARGET])),
).sort_values("mae", ascending=False).round(3)

## 5. Read the worst mistakes

This is the most valuable cell. For each row, ask: what did the experts see that none of the features measure? Each answer is a candidate feature.

In [ ]:
worst = df.reindex(df["error"].abs().sort_values(ascending=False).index).head(10)
worst[["row_id", "block", TARGET, "oof_rescaled", "copy_ratio", "trigram_copy_ratio", "extracted_text"]]